# TP2 — Predicción y archivo de entrega

Usar el **mismo modelo seleccionado y validado** en las notebooks 04 y 05. No volver a entrenarlo aquí.

In [ ]:
from pathlib import Path
import pandas as pd
import joblib

# Rutas del proyecto
ruta_datos = Path("/content/drive/MyDrive/DiploIA/TP2/1-data/2-processed")
ruta_modelos = Path("/content/drive/MyDrive/DiploIA/TP2/2-models")

# Cargar datos procesados para predicción
X_pred = pd.read_csv(ruta_datos / "X_pred.csv")

# Cargar Random Forest seleccionado
modelo_final = joblib.load(
    ruta_modelos / "random_forest_final.pkl"
)

# Verificar datos y modelo
print("Dimensiones de X_pred:", X_pred.shape)
print("Variables utilizadas:", X_pred.shape[1])
print("Modelo:", type(modelo_final).__name__)

# Verificar las variables antes de predecir

In [ ]:
# Verificar que las variables coincidan con el entrenamiento
variables_modelo = list(modelo_final.feature_names_in_)
variables_pred = list(X_pred.columns)

print("¿Coinciden las variables y su orden?",
      variables_modelo == variables_pred)

print("Valores faltantes:", X_pred.isnull().sum().sum())

print("Cantidad de variables del modelo:",
      modelo_final.n_features_in_)

print("Cantidad de variables en X_pred:",
      X_pred.shape[1])
assert variables_modelo == variables_pred, "Las columnas no coinciden con las del entrenamiento"
assert X_pred.isnull().sum().sum() == 0, "Hay valores faltantes"


# Generar las predicciones

In [ ]:
y_pred = modelo_final.predict(X_pred)
distribucion = pd.Series(y_pred).value_counts().sort_index()
porcentajes = (
    pd.Series(y_pred)
    .value_counts(normalize=True)
    .sort_index() * 100)

print("Total de predicciones:", len(y_pred))

print("\nDistribución de predicciones:")

for clase in distribucion.index:
    etiqueta = "No fumador" if clase == 0 else "Fumador"

    print(
        f"{etiqueta} ({clase}): "
        f"{distribucion[clase]} personas "
        f"({porcentajes[clase]:.2f}%)")

# Verificar los identificadores originales

In [ ]:
# Ruta correcta del archivo original para la entrega
ruta_original = Path(
    "/content/drive/MyDrive/DiploIA/TP2/1-data/1-Raw/smoking_prediction_entrega.csv"
)

datos_originales = pd.read_csv(ruta_original)

print("Dimensiones:", datos_originales.shape)
print("¿Contiene smoking?:", "smoking" in datos_originales.columns)
print("¿Coinciden las filas?:", len(datos_originales) == len(y_pred))
print("Primeros ID:", datos_originales["ID"].head().tolist())

In [ ]:
# Recuperar variables originales y codificar las categóricas igual que en el preprocesamiento
X_original = datos_originales.drop(columns=["ID", "oral"]).copy()
X_original["gender"] = X_original["gender"].map({"F": 0, "M": 1})
X_original["tartar"] = X_original["tartar"].map({"N": 0, "Y": 1})
print("Variables originales codificadas:", X_original.shape)

# Verificar el orden de todos los registros
comparar las seis variables que no fueron escaladas, en todas las filas

In [ ]:
# Variables que no fueron escaladas
columnas_sin_escalar = [
    "gender",
    "hearing(left)",
    "hearing(right)",
    "Urine protein",
    "dental caries",
    "tartar"
]

# Comparar los valores de todas las filas
comparacion = (
    X_original[columnas_sin_escalar].reset_index(drop=True)
    ==
    X_pred[columnas_sin_escalar].reset_index(drop=True)
)

print("Coincidencia por variable:")
print(comparacion.all())

print("\nFilas que coinciden completamente:")
print(comparacion.all(axis=1).sum(), "de", len(X_pred))

print("\n¿Coinciden todas las filas?")
print(comparacion.all().all())

# Verificación definitiva del preprocesamiento

In [ ]:
import numpy as np

# Cargar la configuración original del preprocesamiento
preprocesamiento = joblib.load(
    ruta_modelos / "preprocesamiento.joblib"
)

# Recuperar el scaler y las columnas originales
scaler = preprocesamiento["scaler"]
columnas_escaladas = preprocesamiento["columnas_escalar"]

# Copiar los datos originales ya codificados
X_verificacion = X_original.copy()

# Aplicar exactamente el mismo escalado
X_verificacion[columnas_escaladas] = scaler.transform(
    X_verificacion[columnas_escaladas]
)

# Mantener el mismo orden de columnas que X_pred
X_verificacion = X_verificacion[X_pred.columns]

# Comparar los dos conjuntos completos
coinciden = np.allclose(
    X_verificacion.to_numpy(dtype=float),
    X_pred.to_numpy(dtype=float),
    rtol=1e-5,
    atol=1e-8
)

diferencia_maxima = np.max(
    np.abs(
        X_verificacion.to_numpy(dtype=float)
        - X_pred.to_numpy(dtype=float)
    )
)

print("Filas verificadas:", len(X_verificacion))
print("Variables verificadas:", X_verificacion.shape[1])
print("¿Coincide todo el preprocesamiento?:", coinciden)
print("Diferencia máxima:", diferencia_maxima)

# Generar y guardar el CSV final

In [ ]:
# Crear DataFrame con los ID originales y las predicciones
df_predicciones = pd.DataFrame({
    "ID": datos_originales["ID"].astype(int).values,
    "smoking": y_pred.astype(int)
})

# Verificaciones de integridad
assert len(df_predicciones) == len(datos_originales)
assert df_predicciones["ID"].is_unique
assert df_predicciones["smoking"].isin([0, 1]).all()
assert not df_predicciones.isnull().any().any()

# Crear carpeta de resultados
ruta_resultados = Path(
    "/content/drive/MyDrive/DiploIA/TP2/3-results"
)
ruta_resultados.mkdir(parents=True, exist_ok=True)

# Guardar archivo final
ruta_salida = ruta_resultados / "smoking_predicciones_final.csv"

df_predicciones.to_csv(ruta_salida, index=False)

print("Archivo guardado correctamente:")
print(ruta_salida)

print("\nDimensiones:", df_predicciones.shape)
print("\nPrimeras 10 predicciones:")
display(df_predicciones.head(10))

print("\nDistribución de predicciones:")
print(df_predicciones["smoking"].value_counts())

# Verificar el CSV guardado

In [ ]:
df_verificacion = pd.read_csv(ruta_salida)
assert df_verificacion.shape == (5692, 2)
assert df_verificacion.columns.tolist() == ["ID", "smoking"]
assert df_verificacion.equals(df_predicciones)
print("Verificación final: CORRECTA")
print("El archivo guardado coincide con las predicciones.")
print("Cantidad de registros:", len(df_verificacion))
print("Columnas:", df_verificacion.columns.tolist())
print("Archivo:", ruta_salida)